# 05 LIME Interpretation – Configurable Pipeline

## Purpose

This notebook generates LIME explanations using the same standardised approach across:

- German Credit
- Pima Diabetes
- COMPAS

Only the `ACTIVE_DATASET` value in the configuration block should be changed.

The notebook explains all test instances using estimator-only models from `03_baseline_modeling`.

Outputs from this notebook are used by:

- `06_shap_lime_explaination_disagreement`
- `08_01_ees_framework_prepare_inputs`
- `08_02_ees_framework_fidelity_score`
- `08_07_results_consolidation`

## Key Design Principle

LIME creates local surrogate explanations by perturbing observations around each test instance and fitting a simple local model.

To support reproducibility:

- `RANDOM_STATE = 42` is fixed;
- NumPy seed is fixed;
- `LimeTabularExplainer(random_state=RANDOM_STATE)` is used;
- `LIME_NUM_SAMPLES` is fixed across datasets.

The LIME score is interpreted as local surrogate fidelity, meaning how well the local interpretable model approximates the black-box model around a specific instance.

In [14]:
# =========================================================
# 0. Dataset Configuration
# =========================================================

# Select only one active dataset.
# ACTIVE_DATASET = "german_credit"
ACTIVE_DATASET = "pima_diabetes"
# ACTIVE_DATASET = "compas"

DATASET_CONFIGS = {
    "german_credit": {
        "dataset_name": "german_credit",
        "dataset_label": "German Credit",
        "target_name": "target",
        "positive_class_label": "Default",
        "negative_class_label": "Non-default",
        "results_folder": "german_credit",
        "preprocessing_folder": "02_data_preprocessing_german_credit",
        "baseline_folder": "03_baseline_modeling_german_credit",
        "lime_folder": "05_lime_interpretation_german_credit",
        "unfavourable_class": 1,
        "desired_class": 0,
        "immutable_features": ["Age", "Sex"]
    },
    "pima_diabetes": {
        "dataset_name": "pima_diabetes",
        "dataset_label": "Pima Diabetes",
        "target_name": "target",
        "positive_class_label": "Diabetes",
        "negative_class_label": "No diabetes",
        "results_folder": "pima_diabetes",
        "preprocessing_folder": "02_data_preprocessing_pima_diabetes",
        "baseline_folder": "03_baseline_modeling_pima_diabetes",
        "lime_folder": "05_lime_interpretation_pima_diabetes",
        "unfavourable_class": 1,
        "desired_class": 0,
        "immutable_features": ["Age"]
    },
    "compas": {
        "dataset_name": "compas",
        "dataset_label": "COMPAS",
        "target_name": "target",
        "positive_class_label": "Recidivism",
        "negative_class_label": "No recidivism",
        "results_folder": "compas",
        "preprocessing_folder": "02_data_preprocessing_compas",
        "baseline_folder": "03_baseline_modeling_compas",
        "lime_folder": "05_lime_interpretation_compas",
        "unfavourable_class": 1,
        "desired_class": 0,
        "immutable_features": ["age", "sex", "race"]
    }
}

CONFIG = DATASET_CONFIGS[ACTIVE_DATASET]

print("Active dataset:", CONFIG["dataset_label"])
print("Dataset name:", CONFIG["dataset_name"])
print("Positive class:", CONFIG["positive_class_label"])

Active dataset: Pima Diabetes
Dataset name: pima_diabetes
Positive class: Diabetes


In [15]:
# =========================================================
# 1. Install Required Packages
# =========================================================

try:
    from lime.lime_tabular import LimeTabularExplainer
    LIME_AVAILABLE = True
except ImportError:
    print("Installing LIME...")
    !pip install -q lime
    from lime.lime_tabular import LimeTabularExplainer
    LIME_AVAILABLE = True

print("LIME available:", LIME_AVAILABLE)

LIME available: True


In [16]:
# =========================================================
# 2. Import Required Libraries
# =========================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from datetime import datetime
import json
import time
import numpy as np
import pandas as pd
import joblib

pd.set_option("display.max_columns", 250)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
rng = np.random.RandomState(RANDOM_STATE)

print("Libraries imported successfully.")
print("Random seed:", RANDOM_STATE)

Libraries imported successfully.
Random seed: 42


In [17]:
# =========================================================
# 3. Mount Google Drive and Configure Paths
# =========================================================

from google.colab import drive

try:
    drive.mount('/content/drive')
except Exception as e:
    print("Drive mount skipped or already available:", e)

ROOT_PATH = Path("/content/drive/MyDrive/Master_Research")

DATASET_NAME = CONFIG["dataset_name"]
NOTEBOOK_NAME = CONFIG["lime_folder"]

RESULTS_ROOT = ROOT_PATH / "07_Results" / CONFIG["results_folder"]

PREPROCESSING_RESULTS_PATH = RESULTS_ROOT / CONFIG["preprocessing_folder"]
PREPROCESSING_DATASETS_PATH = PREPROCESSING_RESULTS_PATH / "datasets"

BASELINE_RESULTS_PATH = RESULTS_ROOT / CONFIG["baseline_folder"]
BASELINE_MODELS_PATH = BASELINE_RESULTS_PATH / "models"

RESULTS_PATH = RESULTS_ROOT / NOTEBOOK_NAME
TABLES_PATH = RESULTS_PATH / "tables"
DATASETS_PATH = RESULTS_PATH / "datasets"
FIGURES_PATH = RESULTS_PATH / "figures"
LOGS_PATH = RESULTS_PATH / "logs"

for path in [RESULTS_PATH, TABLES_PATH, DATASETS_PATH, FIGURES_PATH, LOGS_PATH]:
    path.mkdir(parents=True, exist_ok=True)

print("Results root:", RESULTS_ROOT)
print("Baseline models path:", BASELINE_MODELS_PATH)
print("LIME output path:", RESULTS_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Results root: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes
Baseline models path: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/03_baseline_modeling_pima_diabetes/models
LIME output path: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/05_lime_interpretation_pima_diabetes


In [18]:
# =========================================================
# 4. Helper Functions
# =========================================================

def read_csv_safely(path, label, required=True):
    path = Path(path)
    if not path.exists():
        if required:
            raise FileNotFoundError(f"{label} not found at: {path}")
        print(f"Optional file not found: {label} at {path}")
        return pd.DataFrame()
    return pd.read_csv(path)


def load_target_series(path):
    target_df = read_csv_safely(path, "target file")
    if "target" in target_df.columns:
        return target_df["target"].astype(int)
    if CONFIG["target_name"] in target_df.columns:
        return target_df[CONFIG["target_name"]].astype(int)
    if target_df.shape[1] == 1:
        return target_df.iloc[:, 0].astype(int)
    raise ValueError(f"Could not identify target column in {path}. Available columns: {target_df.columns.tolist()}")


def save_df(df, filename, folder=None, index=False):
    if folder is None:
        folder = TABLES_PATH
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    output_path = folder / filename
    df.to_csv(output_path, index=index)
    print(f"Saved: {output_path}")
    return output_path


def save_excel(dfs, filename, folder=None):
    if folder is None:
        folder = TABLES_PATH
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    output_path = folder / filename
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for sheet_name, df in dfs.items():
            if isinstance(df, pd.DataFrame):
                df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
    print(f"Saved: {output_path}")
    return output_path


def get_model_probabilities(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X)
        return 1 / (1 + np.exp(-scores))
    preds = model.predict(X)
    return preds.astype(float)


def make_predict_proba_fn(model, feature_names):
    def predict_fn(data):
        data_df = pd.DataFrame(data, columns=feature_names)
        return model.predict_proba(data_df)
    return predict_fn


def extract_lime_feature_rows(explanation, model_id, model_name, test_instance_position):
    rows = []
    for feature_text, weight in explanation.as_list(label=1):
        rows.append({
            "model_id": model_id,
            "model_name": model_name,
            "test_instance_position": test_instance_position,
            "feature_rule": feature_text,
            "feature": feature_text,
            "lime_weight": float(weight),
            "abs_lime_weight": float(abs(weight))
        })
    return rows

## 5. Load Modelling Data

The LIME explainer is fitted on the preprocessed training matrix and explains the preprocessed test matrix.

Tracking/index columns are removed so that LIME explains only model features.

In [19]:
# =========================================================
# 5. Load Modelling Data
# =========================================================

X_train_preprocessed = read_csv_safely(PREPROCESSING_DATASETS_PATH / "X_train_preprocessed.csv", "X_train_preprocessed")
X_test_preprocessed = read_csv_safely(PREPROCESSING_DATASETS_PATH / "X_test_preprocessed.csv", "X_test_preprocessed")
y_train = load_target_series(PREPROCESSING_DATASETS_PATH / "y_train.csv")
y_test = load_target_series(PREPROCESSING_DATASETS_PATH / "y_test.csv")

PREPROCESSED_TRACKING_COLUMNS = ["original_index", "index", "Unnamed: 0"]

X_train_preprocessed = X_train_preprocessed.drop(columns=PREPROCESSED_TRACKING_COLUMNS, errors="ignore").copy()
X_test_preprocessed = X_test_preprocessed.drop(columns=PREPROCESSED_TRACKING_COLUMNS, errors="ignore").copy()

feature_names = X_train_preprocessed.columns.tolist()

print("X_train_preprocessed:", X_train_preprocessed.shape)
print("X_test_preprocessed:", X_test_preprocessed.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)
print("Number of features:", len(feature_names))

display(X_train_preprocessed.head())

X_train_preprocessed: (614, 8)
X_test_preprocessed: (154, 8)
y_train: (614,)
y_test: (154,)
Number of features: 8


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,-0.400000,-0.658537,-0.625000,-2.428571,-8.200000,-0.585915,0.500951,-0.294118
1,0.400000,0.219512,0.375000,-0.285714,-10.300000,-0.315493,0.143310,0.647059
2,-0.200000,-0.292683,-0.875000,1.571429,-3.100000,0.281690,-0.399493,-0.235294
3,-0.400000,0.707317,-1.000000,0.000000,0.000000,-0.304225,0.460368,0.000000
4,-0.600000,-0.536585,-0.500000,1.428571,-2.000000,1.374648,-0.041852,-0.411765


In [20]:
# =========================================================
# 6. Validate LIME Inputs
# =========================================================

validation_rows = [
    {"check": "X_train_preprocessed aligns with y_train", "passed": len(X_train_preprocessed) == len(y_train)},
    {"check": "X_test_preprocessed aligns with y_test", "passed": len(X_test_preprocessed) == len(y_test)},
    {"check": "Train/test columns match", "passed": list(X_train_preprocessed.columns) == list(X_test_preprocessed.columns)},
    {"check": "No missing train values", "passed": X_train_preprocessed.isna().sum().sum() == 0},
    {"check": "No missing test values", "passed": X_test_preprocessed.isna().sum().sum() == 0},
    {"check": "Binary test target", "passed": sorted(pd.Series(y_test).unique().tolist()) == [0, 1]},
]

lime_input_validation_df = pd.DataFrame(validation_rows)

display(lime_input_validation_df)

save_df(lime_input_validation_df, "lime_input_validation.csv")

if not lime_input_validation_df["passed"].all():
    raise ValueError("One or more LIME input validation checks failed.")

,check,passed
0,X_train_preprocessed aligns with y_train,True
1,X_test_preprocessed aligns with y_test,True
2,Train/test columns match,True
3,No missing train values,True
4,No missing test values,True
5,Binary test target,True


Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/05_lime_interpretation_pima_diabetes/tables/lime_input_validation.csv


## 7. Load Estimator-Only Models

LIME explains the estimator-only models saved by `03_baseline_modeling`.

In [21]:
# =========================================================
# 7. Load Estimator-Only Models
# =========================================================

model_files = {
    "logistic_regression": "logistic_regression.pkl",
    "decision_tree": "decision_tree.pkl",
    "random_forest": "random_forest.pkl",
    "xgboost": "xgboost.pkl",
    "mlp": "mlp.pkl",
}

pretty_model_names = {
    "logistic_regression": "Logistic Regression",
    "decision_tree": "Decision Tree",
    "random_forest": "Random Forest",
    "xgboost": "XGBoost",
    "mlp": "MLP",
}

models = {}
model_load_rows = []

for model_id, filename in model_files.items():
    model_path = BASELINE_MODELS_PATH / filename

    if model_path.exists():
        models[model_id] = joblib.load(model_path)
        found = True
        status = "Loaded"
    else:
        found = False
        status = "Skipped - model file not found"

    model_load_rows.append({
        "model_id": model_id,
        "model_name": pretty_model_names[model_id],
        "model_file_found": found,
        "model_file_path": str(model_path),
        "status": status
    })

model_load_registry_df = pd.DataFrame(model_load_rows)

display(model_load_registry_df)

save_df(model_load_registry_df, "lime_model_load_registry.csv")

if not models:
    raise ValueError("No estimator-only models were loaded.")

,model_id,model_name,model_file_found,model_file_path,status
0,logistic_regression,Logistic Regression,True,/content/drive/MyDrive/Master_Research/07_Resu...,Loaded
1,decision_tree,Decision Tree,True,/content/drive/MyDrive/Master_Research/07_Resu...,Loaded
2,random_forest,Random Forest,True,/content/drive/MyDrive/Master_Research/07_Resu...,Loaded
3,xgboost,XGBoost,True,/content/drive/MyDrive/Master_Research/07_Resu...,Loaded
4,mlp,MLP,True,/content/drive/MyDrive/Master_Research/07_Resu...,Loaded


Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/05_lime_interpretation_pima_diabetes/tables/lime_model_load_registry.csv


## 8. LIME Configuration

`LIME_NUM_SAMPLES` controls the number of perturbed neighbourhood samples used to fit the local surrogate model.

`NUM_FEATURES` controls how many features are retained in each local explanation.

For consistency with SHAP-LIME comparison, `NUM_FEATURES = 10` is used for the local explanation ranking.

In [22]:
# =========================================================
# 8. LIME Configuration
# =========================================================

NUM_FEATURES = 10
LIME_NUM_SAMPLES = 5000

class_names = [CONFIG["negative_class_label"], CONFIG["positive_class_label"]]

lime_config_df = pd.DataFrame([
    {"parameter": "RANDOM_STATE", "value": RANDOM_STATE, "description": "Fixed seed used for reproducible LIME perturbation sampling."},
    {"parameter": "NUM_FEATURES", "value": NUM_FEATURES, "description": "Number of local explanation features retained per instance."},
    {"parameter": "LIME_NUM_SAMPLES", "value": LIME_NUM_SAMPLES, "description": "Number of perturbed samples used by LIME for local surrogate fitting."},
    {"parameter": "explanation_population", "value": "all_test_instances", "description": "LIME explanations are generated for all test instances."}
])

display(lime_config_df)

save_df(lime_config_df, "lime_config.csv")

,parameter,value,description
0,RANDOM_STATE,42,Fixed seed used for reproducible LIME perturba...
1,NUM_FEATURES,10,Number of local explanation features retained ...
2,LIME_NUM_SAMPLES,5000,Number of perturbed samples used by LIME for l...
3,explanation_population,all_test_instances,LIME explanations are generated for all test i...


Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/05_lime_interpretation_pima_diabetes/tables/lime_config.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/05_lime_interpretation_pima_diabetes/tables/lime_config.csv')

## 9. Create Reproducible LIME Explainer

The explainer is initialised with `random_state=42`.

This controls LIME's perturbation sampling and supports reproducible feature weights and local surrogate fidelity scores.

In [23]:
# =========================================================
# 9. Create Reproducible LIME Explainer
# =========================================================

lime_explainer = LimeTabularExplainer(
    training_data=X_train_preprocessed.values,
    feature_names=feature_names,
    class_names=class_names,
    mode="classification",
    discretize_continuous=True,
    random_state=RANDOM_STATE
)

lime_explainer_summary_df = pd.DataFrame([{
    "dataset_name": DATASET_NAME,
    "n_training_rows": X_train_preprocessed.shape[0],
    "n_features": X_train_preprocessed.shape[1],
    "mode": "classification",
    "discretize_continuous": True,
    "random_state": RANDOM_STATE,
    "class_0": class_names[0],
    "class_1": class_names[1]
}])

display(lime_explainer_summary_df)

save_df(lime_explainer_summary_df, "lime_explainer_summary.csv")

,dataset_name,n_training_rows,n_features,mode,discretize_continuous,random_state,class_0,class_1
0,pima_diabetes,614,8,classification,True,42,No diabetes,Diabetes


Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/05_lime_interpretation_pima_diabetes/tables/lime_explainer_summary.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/05_lime_interpretation_pima_diabetes/tables/lime_explainer_summary.csv')

## 10. Generate All-Test LIME Explanations

This section creates:

- local LIME feature weights for every test instance;
- model prediction summaries;
- LIME local surrogate fidelity scores;
- global LIME importance summaries.

In [24]:
# =========================================================
# 10. Generate All-Test LIME Explanations
# =========================================================

lime_long_rows = []
prediction_rows = []
lime_fidelity_rows = []
global_importance_rows = []
error_rows = []
runtime_rows = []

X_test_values = X_test_preprocessed.values

for model_id, model in models.items():

    model_name = pretty_model_names[model_id]
    predict_fn = make_predict_proba_fn(model, feature_names)

    print("=" * 100)
    print(f"Generating LIME explanations for: {model_name}")
    print("=" * 100)

    start_time = time.time()

    try:
        y_pred = model.predict(X_test_preprocessed)
        y_proba = get_model_probabilities(model, X_test_preprocessed)

        model_lime_rows = []

        for i in range(len(X_test_preprocessed)):
            explanation = lime_explainer.explain_instance(
                data_row=X_test_values[i],
                predict_fn=predict_fn,
                labels=(1,),
                num_features=NUM_FEATURES,
                num_samples=LIME_NUM_SAMPLES
            )

            local_prediction = np.nan
            try:
                local_prediction_values = explanation.local_pred
                local_prediction = float(np.ravel(local_prediction_values)[0])
            except Exception:
                local_prediction = np.nan

            lime_r2_score = float(explanation.score) if explanation.score is not None else np.nan
            lime_local_error = abs(float(y_proba[i]) - local_prediction) if not pd.isna(local_prediction) else np.nan

            prediction_rows.append({
                "model_id": model_id,
                "model_name": model_name,
                "test_instance_position": i,
                "true_class": int(y_test.iloc[i] if hasattr(y_test, "iloc") else y_test[i]),
                "predicted_class": int(y_pred[i]),
                "predicted_probability": float(y_proba[i])
            })

            lime_fidelity_rows.append({
                "model_id": model_id,
                "model_name": model_name,
                "test_instance_position": i,
                "predicted_probability": float(y_proba[i]),
                "lime_local_prediction": local_prediction,
                "lime_r2_score": lime_r2_score,
                "lime_local_error": lime_local_error,
                "num_features": NUM_FEATURES,
                "num_samples": LIME_NUM_SAMPLES,
                "random_state": RANDOM_STATE
            })

            instance_rows = extract_lime_feature_rows(explanation, model_id, model_name, i)
            model_lime_rows.extend(instance_rows)
            lime_long_rows.extend(instance_rows)

        elapsed_seconds = time.time() - start_time

        runtime_rows.append({
            "model_id": model_id,
            "model_name": model_name,
            "elapsed_seconds": elapsed_seconds,
            "n_test_instances_explained": len(X_test_preprocessed),
            "num_features": NUM_FEATURES,
            "num_samples": LIME_NUM_SAMPLES
        })

        model_lime_df = pd.DataFrame(model_lime_rows)
        if not model_lime_df.empty:
            model_global_df = (
                model_lime_df
                .groupby(["model_id", "model_name", "feature_rule"], as_index=False)
                .agg(
                    mean_abs_lime_weight=("abs_lime_weight", "mean"),
                    mean_lime_weight=("lime_weight", "mean"),
                    n_instances=("test_instance_position", "nunique")
                )
            )
            global_importance_rows.extend(model_global_df.to_dict("records"))

        print(f"Completed {model_name} in {elapsed_seconds:.2f} seconds.")

    except Exception as e:
        error_rows.append({"model_id": model_id, "model_name": model_name, "error_message": str(e)})
        print(f"LIME failed for {model_name}: {e}")

lime_all_test_long_df = pd.DataFrame(lime_long_rows)
lime_prediction_summary_df = pd.DataFrame(prediction_rows)
lime_fidelity_inputs_df = pd.DataFrame(lime_fidelity_rows)
global_lime_importance_summary_df = pd.DataFrame(global_importance_rows)
lime_runtime_diagnostics_df = pd.DataFrame(runtime_rows)
lime_error_log_df = pd.DataFrame(error_rows)

if not global_lime_importance_summary_df.empty:
    global_lime_importance_summary_df = global_lime_importance_summary_df.sort_values(["model_id", "mean_abs_lime_weight"], ascending=[True, False]).reset_index(drop=True)

print("LIME long shape:", lime_all_test_long_df.shape)
print("Prediction summary shape:", lime_prediction_summary_df.shape)
print("Fidelity inputs shape:", lime_fidelity_inputs_df.shape)
print("Global LIME importance shape:", global_lime_importance_summary_df.shape)
print("Runtime diagnostics shape:", lime_runtime_diagnostics_df.shape)
print("Error log shape:", lime_error_log_df.shape)

display(lime_all_test_long_df.head())
display(lime_fidelity_inputs_df.head())
display(global_lime_importance_summary_df.head())
display(lime_error_log_df)

Generating LIME explanations for: Logistic Regression
Completed Logistic Regression in 4.40 seconds.
Generating LIME explanations for: Decision Tree
Completed Decision Tree in 3.93 seconds.
Generating LIME explanations for: Random Forest
Completed Random Forest in 27.24 seconds.
Generating LIME explanations for: XGBoost
Completed XGBoost in 8.23 seconds.
Generating LIME explanations for: MLP
Completed MLP in 6.75 seconds.
LIME long shape: (6160, 7)
Prediction summary shape: (770, 6)
Fidelity inputs shape: (770, 10)
Global LIME importance shape: (160, 6)
Runtime diagnostics shape: (5, 6)
Error log shape: (0, 0)


,model_id,model_name,test_instance_position,feature_rule,feature,lime_weight,abs_lime_weight
0,logistic_regression,Logistic Regression,0,Glucose > 0.56,Glucose > 0.56,0.428573,0.428573
1,logistic_regression,Logistic Regression,0,BMI <= -0.54,BMI <= -0.54,-0.175981,0.175981
2,logistic_regression,Logistic Regression,0,Pregnancies > 0.60,Pregnancies > 0.60,0.125933,0.125933
3,logistic_regression,Logistic Regression,0,-0.35 < DiabetesPedigreeFunction <= 0.00,-0.35 < DiabetesPedigreeFunction <= 0.00,-0.030565,0.030565
4,logistic_regression,Logistic Regression,0,0.00 < Age <= 0.71,0.00 < Age <= 0.71,0.015341,0.015341


,model_id,model_name,test_instance_position,predicted_probability,lime_local_prediction,lime_r2_score,lime_local_error,num_features,num_samples,random_state
0,logistic_regression,Logistic Regression,0,0.604280,0.640805,0.662065,0.036525,10,5000,42
1,logistic_regression,Logistic Regression,1,0.120747,0.256138,0.343324,0.135391,10,5000,42
2,logistic_regression,Logistic Regression,2,0.294740,0.320333,0.009619,0.025593,10,5000,42
3,logistic_regression,Logistic Regression,3,0.251213,0.295407,0.236186,0.044194,10,5000,42
4,logistic_regression,Logistic Regression,4,0.034765,0.006260,0.304772,0.028505,10,5000,42


,model_id,model_name,feature_rule,mean_abs_lime_weight,mean_lime_weight,n_instances
0,decision_tree,Decision Tree,Glucose > 0.56,0.390406,0.390406,39
1,decision_tree,Decision Tree,BMI <= -0.54,0.269000,-0.269000,46
2,decision_tree,Decision Tree,Glucose <= -0.44,0.241355,-0.241355,37
3,decision_tree,Decision Tree,BMI > 0.46,0.165955,0.165955,41
4,decision_tree,Decision Tree,-0.44 < Glucose <= 0.00,0.149712,-0.149712,40


""


## 11. LIME Fidelity Diagnostic Summary

LIME fidelity is based on local surrogate approximation quality.

The key fields are:

- `lime_r2_score`: local surrogate goodness-of-fit;
- `lime_local_error`: absolute difference between the black-box probability and LIME local surrogate prediction.

These are used by the EES Fidelity pillar.

In [25]:
# =========================================================
# 11. LIME Fidelity Diagnostic Summary
# =========================================================

if lime_fidelity_inputs_df.empty:
    lime_fidelity_summary_df = pd.DataFrame()
else:
    lime_fidelity_summary_df = (
        lime_fidelity_inputs_df
        .groupby(["model_id", "model_name"], as_index=False)
        .agg(
            n_instances=("test_instance_position", "nunique"),
            mean_lime_r2_score=("lime_r2_score", "mean"),
            median_lime_r2_score=("lime_r2_score", "median"),
            min_lime_r2_score=("lime_r2_score", "min"),
            max_lime_r2_score=("lime_r2_score", "max"),
            mean_lime_local_error=("lime_local_error", "mean"),
            median_lime_local_error=("lime_local_error", "median"),
            max_lime_local_error=("lime_local_error", "max")
        )
    )

display(lime_fidelity_summary_df)

save_df(lime_fidelity_summary_df, "lime_fidelity_summary.csv")

,model_id,model_name,n_instances,mean_lime_r2_score,median_lime_r2_score,min_lime_r2_score,max_lime_r2_score,mean_lime_local_error,median_lime_local_error,max_lime_local_error
0,decision_tree,Decision Tree,154,0.181610,0.167189,0.015059,0.436236,0.155058,0.126229,0.733651
1,logistic_regression,Logistic Regression,154,0.317475,0.274653,0.009619,0.716388,0.076030,0.065765,0.262418
2,mlp,MLP,154,0.290141,0.252535,0.015515,0.627714,0.084704,0.071775,0.485406
3,random_forest,Random Forest,154,0.276355,0.243846,0.031737,0.625853,0.081891,0.066744,0.338057
4,xgboost,XGBoost,154,0.252181,0.191959,0.013031,0.585645,0.126500,0.110441,0.519993


Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/05_lime_interpretation_pima_diabetes/tables/lime_fidelity_summary.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/05_lime_interpretation_pima_diabetes/tables/lime_fidelity_summary.csv')

## 12. Save LIME Outputs

In [26]:
# =========================================================
# 12. Save LIME Outputs
# =========================================================

save_df(lime_all_test_long_df, "lime_all_test_long.csv")
save_df(lime_prediction_summary_df, "lime_prediction_summary.csv")
save_df(lime_fidelity_inputs_df, "lime_fidelity_inputs.csv")
save_df(lime_fidelity_summary_df, "lime_fidelity_summary.csv")
save_df(global_lime_importance_summary_df, "global_lime_importance_summary.csv")
save_df(lime_runtime_diagnostics_df, "lime_runtime_diagnostics.csv")
save_df(lime_error_log_df, "lime_error_log.csv")

save_df(lime_all_test_long_df, "lime_all_test_long.csv", folder=DATASETS_PATH)
save_df(lime_prediction_summary_df, "lime_prediction_summary.csv", folder=DATASETS_PATH)
save_df(lime_fidelity_inputs_df, "lime_fidelity_inputs.csv", folder=DATASETS_PATH)
save_df(lime_fidelity_summary_df, "lime_fidelity_summary.csv", folder=DATASETS_PATH)
save_df(global_lime_importance_summary_df, "global_lime_importance_summary.csv", folder=DATASETS_PATH)
save_df(lime_runtime_diagnostics_df, "lime_runtime_diagnostics.csv", folder=DATASETS_PATH)
save_df(lime_error_log_df, "lime_error_log.csv", folder=DATASETS_PATH)

save_excel(
    {
        "lime_all_test_long": lime_all_test_long_df,
        "prediction_summary": lime_prediction_summary_df,
        "fidelity_inputs": lime_fidelity_inputs_df,
        "fidelity_summary": lime_fidelity_summary_df,
        "global_importance": global_lime_importance_summary_df,
        "runtime_diagnostics": lime_runtime_diagnostics_df,
        "model_load_registry": model_load_registry_df,
        "input_validation": lime_input_validation_df,
        "lime_config": lime_config_df,
        "explainer_summary": lime_explainer_summary_df,
        "error_log": lime_error_log_df
    },
    f"lime_interpretation_{DATASET_NAME}_outputs.xlsx"
)

manifest = {
    "dataset_name": DATASET_NAME,
    "dataset_label": CONFIG["dataset_label"],
    "notebook_name": NOTEBOOK_NAME,
    "run_timestamp": datetime.now().isoformat(),
    "random_state": RANDOM_STATE,
    "positive_class_label": CONFIG["positive_class_label"],
    "negative_class_label": CONFIG["negative_class_label"],
    "explanation_population": "all_test_instances",
    "num_features": NUM_FEATURES,
    "lime_num_samples": LIME_NUM_SAMPLES,
    "lime_random_state": RANDOM_STATE,
    "models_loaded": list(models.keys()),
    "standardisation_note": "This LIME notebook uses the same configurable pipeline, seeded explainer and output structure across German Credit, Pima Diabetes and COMPAS."
}

with open(LOGS_PATH / "lime_interpretation_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

with open(TABLES_PATH / "lime_interpretation_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

print("All LIME outputs saved.")

Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/05_lime_interpretation_pima_diabetes/tables/lime_all_test_long.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/05_lime_interpretation_pima_diabetes/tables/lime_prediction_summary.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/05_lime_interpretation_pima_diabetes/tables/lime_fidelity_inputs.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/05_lime_interpretation_pima_diabetes/tables/lime_fidelity_summary.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/05_lime_interpretation_pima_diabetes/tables/global_lime_importance_summary.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/05_lime_interpretation_pima_diabetes/tables/lime_runtime_diagnostics.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/05_lime_interpretation_pima_diabetes/tables/lime_error_log.csv
Saved: /co

## 13. Interpretation Note

This notebook provides LIME explanation evidence required for the EES framework.

LIME local surrogate fidelity is used as a complementary fidelity signal to SHAP reconstruction fidelity.

A high LIME fidelity score indicates that the local interpretable surrogate model closely approximates the black-box model around the explained instance.

Because this notebook fixes the random seed and LIME sampling configuration, the outputs are reproducible across reruns.

In [27]:
# =========================================================
# 13. Final Completion Message
# =========================================================

print("=" * 100)
print(f"05 LIME Interpretation completed successfully for: {CONFIG['dataset_label']}")
print("=" * 100)

print(f"Tables saved to:   {TABLES_PATH}")
print(f"Datasets saved to: {DATASETS_PATH}")
print(f"Logs saved to:     {LOGS_PATH}")

print("Key outputs:")
print("- lime_all_test_long.csv")
print("- lime_prediction_summary.csv")
print("- lime_fidelity_inputs.csv")
print("- lime_fidelity_summary.csv")
print("- global_lime_importance_summary.csv")
print("- lime_runtime_diagnostics.csv")
print(f"- lime_interpretation_{DATASET_NAME}_outputs.xlsx")

05 LIME Interpretation completed successfully for: Pima Diabetes
Tables saved to:   /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/05_lime_interpretation_pima_diabetes/tables
Datasets saved to: /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/05_lime_interpretation_pima_diabetes/datasets
Logs saved to:     /content/drive/MyDrive/Master_Research/07_Results/pima_diabetes/05_lime_interpretation_pima_diabetes/logs
Key outputs:
- lime_all_test_long.csv
- lime_prediction_summary.csv
- lime_fidelity_inputs.csv
- lime_fidelity_summary.csv
- global_lime_importance_summary.csv
- lime_runtime_diagnostics.csv
- lime_interpretation_pima_diabetes_outputs.xlsx
